In [3]:
from pathlib import Path

import pandas as pd

DATA = Path("../data") if Path("../data").exists() else Path("data")
FALTANTES = ["", "None", "nan", "NaN", "NULL", "null"]

def leer(nombre):
    return pd.read_csv(DATA / f"{nombre}.csv",
                       dtype=str,
                       keep_default_na=False,
                       encoding="utf-8-sig"
                       )
    
cat_perfil = leer("cat_perfil_riesgo")
cat_activos = leer("catalogo_activos")
cat_banca = leer("catalogo_banca")
m = leer("historico_aba_macroactivos")
u = leer("historico_aba_usd_internacional")



In [4]:
print(DATA.resolve())
print(sorted(p.name for p in DATA.glob("*.csv")))

C:\Users\ASUS\Desktop\Prueba_Técnica\proyecto-analítica-inversiones\data
['cat_perfil_riesgo.csv', 'catalogo_activos.csv', 'catalogo_banca.csv', 'historico_aba_macroactivos.csv', 'historico_aba_usd_internacional.csv']


In [5]:
tablas = {
    "cat_perfil_riesgo": cat_perfil,
    "catalogo_activos": cat_activos,
    "catalogo_banca": cat_banca,
    "historico_aba_macroactivos": m,
    "historico_aba_usd_internacional": u,
}
for nombre, df in tablas.items():
    print(f"{nombre}: {df.shape[0]} filas x {df.shape[1]} columnas")
    print("  columnas:", list(df.columns))

cat_perfil_riesgo: 4 filas x 2 columnas
  columnas: ['cod_perfil_riesgo', 'perfil_riesgo']
catalogo_activos: 20 filas x 2 columnas
  columnas: ['activo', 'cod_activo']
catalogo_banca: 6 filas x 2 columnas
  columnas: ['cod_banca', 'banca']
historico_aba_macroactivos: 6694 filas x 11 columnas
  columnas: ['ingestion_year', 'ingestion_month', 'ingestion_day', 'id_sistema_cliente', 'macroactivo', 'cod_activo', 'aba', 'cod_perfil_riesgo', 'cod_banca', 'year', 'month']
historico_aba_usd_internacional: 5750 filas x 12 columnas
  columnas: ['ingestion_year', 'ingestion_month', 'ingestion_day', 'id_sistema_cliente', 'simbol', 'cusip', 'isin', 'nombre_activo', 'cantidad', 'valor_mercado', 'fecha_vencimiento', 'tasa_cupon']


In [6]:
def faltantes(df):
    f = df.isin(FALTANTES).sum()
    return f[f > 0]
print(faltantes(m))
print(faltantes(u))

ingestion_month         4
id_sistema_cliente      3
macroactivo             7
cod_activo            230
aba                     6
cod_perfil_riesgo     119
cod_banca              10
year                    1
month                   3
dtype: int64
simbol        2509
tasa_cupon       2
dtype: int64


In [8]:
for col in ["macroactivo", "cod_perfil_riesgo", "cod_banca", "ingestion_year", "ingestion_month"]:
    print(f"--- {col} ---")
    print(m[col].value_counts(dropna=False).head(15))
    print()

--- macroactivo ---
macroactivo
Renta Variable    3360
FICs              3092
Renta Fija         227
                     7
32184607             3
890112256            1
20203023             1
1018                 1
1002                 1
1005                 1
Name: count, dtype: int64

--- cod_perfil_riesgo ---
cod_perfil_riesgo
1468           3094
1466           2350
1467            899
1469            222
None            112
                  7
PN                4
100931241.5       1
22350729.74       1
78815.51          1
78472.41          1
12615000          1
76142.74          1
Name: count, dtype: int64

--- cod_banca ---
cod_banca
PN      4247
PR      1080
PF       898
PY       334
EG       114
          10
1468       6
2024       4
1466       1
Name: count, dtype: int64

--- ingestion_year ---
ingestion_year
2024    5419
2023    1275
Name: count, dtype: int64

--- ingestion_month ---
ingestion_month
4     1301
2     1257
3     1242
1     1141
12     910
5      474
11     365


In [9]:
raras = m[~m["macroactivo"].isin(["Renta Variable", "FICs", "Renta Fija"] + FALTANTES)]
print(len(raras))
print(raras)

8
     ingestion_year ingestion_month ingestion_day id_sistema_cliente  \
55             2024               3            14                      
102            2024               4            17                      
151            2024               5            10                100   
394            2024               4      39553126            100FICs   
447            2024               1      32184607  100Renta Variable   
492            2024               3      39616197  100Renta Variable   
536            2024               4            25                100   
6659           2024               1             2                100   

     macroactivo cod_activo    aba cod_perfil_riesgo cod_banca year month  
55     890112256       FICs   1008       22350729.74      1468   PR  2024  
102     20203023       FICs  10007       100931241.5      1468   PR  2024  
151     32184607       FICs   1019          78815.51      1468   PN  2024  
394         1018  650060.72   1466           

In [10]:
print(cat_perfil)
print(cat_banca)      # ¿notas algo repetido?
print(cat_activos)    # ¿hay nombres repetidos o códigos raros?

  cod_perfil_riesgo perfil_riesgo
0              1469      AGRESIVO
1              1468      MODERADO
2              1466   SIN DEFINIR
3              1467   CONSERVADOR
  cod_banca         banca
0        PR       Privada
1        PN      Personal
2        PF  Preferencial
3        EG      Empresas
4        PY         Pymes
5        PR       Privada
                         activo cod_activo
0          CDT BANCOLOMBIA B360       1000
1              CDT TUYA TF B360       1001
2                        CELSIA       1002
3                      CEMARGOS       1003
4                     ECOPETROL       1004
5                           ETB       1005
6                    Fiducuenta       1007
7                     Fidurenta       1008
8   Fondo Cerrado Renta Fija IV       1009
9    Fondo Cerrado Renta Fija V       1010
10                   GRUBOLIVAR       1011
11                          ISA       1012
12                     CEMARGOS       1013
13                     PFBCOLOM       1014
14 

In [11]:
aba = pd.to_numeric(m["aba"], errors="coerce")   # lo no numérico pasa a NaN
print("aba no numérico:", aba.isna().sum())
print(aba.describe())
print(m.loc[aba.isna()])                           # mira esas filas

# Extremos: lo muy pequeño o muy grande también es sospechoso
print(m.assign(aba_n=aba).sort_values("aba_n").head(5))
print(m.assign(aba_n=aba).sort_values("aba_n").tail(5))

aba no numérico: 6
count    6.688000e+03
mean     1.105639e+08
std      2.526428e+08
min      1.002000e+03
25%      1.147500e+06
50%      5.233011e+06
75%      3.401826e+07
max      1.533001e+09
Name: aba, dtype: float64
     ingestion_year ingestion_month ingestion_day id_sistema_cliente  \
299            2023              11            23        10038643094   
333            2024               3            11        10041560001   
357            2024               2            14        10039553126   
375            2023              12            19         1.0089E+11   
396            2024               1             2        10032184607   
6660           2024               1            23          1.009E+11   

         macroactivo cod_activo aba cod_perfil_riesgo cod_banca  year month  
299             FICs      10007                  1466        PN  2023    11  
333   Renta Variable       1004                  1466        PN  2024     3  
357             FICs       1018         

In [12]:
vm = pd.to_numeric(u["valor_mercado"], errors="coerce")
print("no numéricos:", vm.isna().sum(), "| negativos:", (vm < 0).sum(), "| ceros:", (vm == 0).sum())
print(u.loc[vm < 0].head())

no numéricos: 2 | negativos: 51 | ceros: 1
     ingestion_year ingestion_month ingestion_day id_sistema_cliente simbol  \
1291           2024               3             1        10014876058    USD   
1347           2024               3             1        10014876058    USD   
1393           2024               3             1        10014876058    USD   
1435           2024               3             1        10014876058    USD   
1621           2024               3             1        10020203023    USD   

          cusip      isin nombre_activo cantidad valor_mercado  \
1291  USD999997  Liquidez          CASH        0        -70000   
1347  USD999997  Liquidez          CASH        0        -50000   
1393  USD999997  Liquidez          CASH        0        -50000   
1435  USD999997  Liquidez          CASH        0       -210000   
1621  USD999997  Liquidez          CASH        0       -143.55   

     fecha_vencimiento tasa_cupon  
1291         1/01/1900          0  
1347         

In [13]:
partes = m[["ingestion_year", "ingestion_month", "ingestion_day"]].apply(pd.to_numeric, errors="coerce")
partes.columns = ["year", "month", "day"]
m["fecha"] = pd.to_datetime(partes, errors="coerce")

print("fechas inválidas:", m["fecha"].isna().sum())
print("rango:", m["fecha"].min(), "->", m["fecha"].max())

fechas inválidas: 7
rango: 2023-11-23 00:00:00 -> 2024-05-15 00:00:00


In [14]:
u["f"] = u["ingestion_year"] + "-" + u["ingestion_month"] + "-" + u["ingestion_day"]
print(u["f"].value_counts())

f
2024-3-1     5548
2024-5-30     102
2024-4-26     100
Name: count, dtype: int64


In [15]:
p = u["fecha_vencimiento"].str.split("/", expand=True)
print("primer campo > 12: ", (pd.to_numeric(p[0], errors="coerce") > 12).sum())
print("segundo campo > 12:", (pd.to_numeric(p[1], errors="coerce") > 12).sum())

primer campo > 12:  1
segundo campo > 12: 877


In [16]:
p = u["fecha_vencimiento"].str.split("/", expand=True)
print("primer campo > 12: ", (pd.to_numeric(p[0], errors="coerce") > 12).sum())
print("segundo campo > 12:", (pd.to_numeric(p[1], errors="coerce") > 12).sum())

primer campo > 12:  1
segundo campo > 12: 877


In [17]:
print(u.loc[u["nombre_activo"].str.contains("BANCOLOMBIA PANAMA SA BOND", na=False),
            ["nombre_activo", "fecha_vencimiento"]].head(3))

                                          nombre_activo fecha_vencimiento
1241  BANCOLOMBIA PANAMA SA BOND 6.100% 11/13/24 REG...        11/13/2024
1244  BANCOLOMBIA PANAMA SA BOND 6.100% 11/13/24 REG...        11/13/2024
1248  BANCOLOMBIA PANAMA SA BOND 6.100% 11/13/24 REG...        11/13/2024


In [18]:
ids = m["id_sistema_cliente"]
print("IDs distintos:", ids.nunique())
print("en notación científica:", ids.str.contains("E+", regex=False).sum())
print(ids.str.len().value_counts().sort_index())
print(sorted(ids[ids.str.contains("E+", regex=False)].unique()))

IDs distintos: 34
en notación científica: 2018
id_sistema_cliente
0        3
3        8
7        1
9      672
10     449
11    5559
17       2
Name: count, dtype: int64
['1.00101E+12', '1.00103E+12', '1.00105E+12', '1.00109E+12', '1.00114E+12', '1.0083E+11', '1.0089E+11', '1.008E+11', '1.00901E+11', '1.00902E+11', '1.009E+11']


In [19]:
print("duplicados exactos macro:", m.duplicated().sum())
print("duplicados exactos usd:  ", u.duplicated().sum())

duplicados exactos macro: 783
duplicados exactos usd:   543


In [20]:
print(u.groupby(["id_sistema_cliente", "f"]).size().unstack(fill_value=0))

f                   2024-3-1  2024-4-26  2024-5-30
id_sistema_cliente                                
10014876058              342         19         20
10020203023              108          2          3
10021382511              710         12         12
10026419826              993         14         14
10032184607              210          4          4
10036346714              193          3          3
10038643094              765         10         10
10039538018              126          2          2
10039553126              264          5          5
10039616197              130          2          2
10041560001              507          8          8
1004870235              1199         19         19
EPMP5694987                1          0          0


In [21]:
x = u[(u["id_sistema_cliente"] == "10014876058") & (u["isin"] == "IE00B7KFL990")]
print(x[["f", "nombre_activo", "cantidad", "valor_mercado"]])

              f                             nombre_activo  cantidad  \
1245   2024-3-1  PIMCO INCOME FUND CLASS E (ACC)(USD)      4162.354   
1249   2024-3-1  PIMCO INCOME FUND CLASS E (ACC)(USD)      4162.354   
1254   2024-3-1  PIMCO INCOME FUND CLASS E (ACC)(USD)      4162.354   
1269   2024-3-1  PIMCO INCOME FUND CLASS E (ACC)(USD)      4162.354   
1270   2024-3-1  PIMCO INCOME FUND CLASS E (ACC)(USD)      4162.354   
...         ...                                       ...       ...   
1561   2024-3-1  PIMCO INCOME FUND CLASS E (ACC)(USD)      4162.354   
1565   2024-3-1  PIMCO INCOME FUND CLASS E (ACC)(USD)      4162.354   
1569   2024-3-1  PIMCO INCOME FUND CLASS E (ACC)(USD)      4162.354   
1583  2024-4-26  PIMCO INCOME FUND CLASS E (ACC)(USD)      4162.354   
1612  2024-5-30  PIMCO INCOME FUND CLASS E (ACC)(USD)      4162.354   

     valor_mercado  
1245      64682.98  
1249      61561.22  
1254      61769.33  
1269       61894.2  
1270       60645.5  
...            ...  


In [22]:
# ¿Los códigos de activo de los datos existen en el catálogo?
en_datos = set(m["cod_activo"]) - set(FALTANTES)
print(sorted(en_datos - set(cat_activos["cod_activo"])))

# ¿Qué clientes están en ambas tablas?
ids_m, ids_u = set(m["id_sistema_cliente"]), set(u["id_sistema_cliente"])
print("en ambas:", len(ids_m & ids_u))
print("solo en USD:", sorted(ids_u - ids_m))

['10007', '1015', '1022', '650060.72', '90535', '9480000', 'FICs', 'Renta Variable']
en ambas: 12
solo en USD: ['EPMP5694987']
